[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/52_rectified_flow_loss_solution.ipynb)

# 🟡 Solution: Rectified Flow Loss

Reference solution for `rectified_flow_loss`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def rectified_flow_loss(model, x0: torch.Tensor, x1: torch.Tensor,
                        t: torch.Tensor | None = None, reduction: str = "mean") -> torch.Tensor:
    B = x0.shape[0]
    if t is None:
        t = torch.rand(B, device=x0.device, dtype=x0.dtype)
    view_shape = (B,) + (1,) * (x0.ndim - 1)
    t_view = t.view(view_shape)
    x_t = (1.0 - t_view) * x0 + t_view * x1
    target = x1 - x0
    pred = model(x_t, t)
    loss = (pred - target).pow(2)
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(f"Unknown reduction: {reduction}")


In [ ]:
# Verify
class ZeroModel:
    def __call__(self, x, t):
        return torch.zeros_like(x)

x0 = torch.zeros(2, 3)
x1 = torch.ones(2, 3)
print(rectified_flow_loss(ZeroModel(), x0, x1, t=torch.tensor([0.25, 0.75])))


In [ ]:
# Run judge
from torch_judge import check
check('rectified_flow_loss')
